<a href="https://colab.research.google.com/github/itsCodingCasper/satellite-image-classification/blob/main/Satellite_image_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastai


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import shutil

shutil.copytree(
    "/content/drive/MyDrive/Study/Satellite img classifier/EuroSAT_RGB",
    "/content/EuroSAT_RGB"
)

'/content/EuroSAT_RGB'

In [4]:
import os

dataset_path = "/content/EuroSAT_RGB"

print(os.listdir(dataset_path))

['Highway', 'AnnualCrop', 'Industrial', 'Forest', 'Residential', 'HerbaceousVegetation', 'PermanentCrop', 'River', 'Pasture', 'SeaLake']


Preprocessing

---
  Resizing: As ResNet50 expects img sizre to be 224x224, whereas currently it is 64x64
  Augmentation: Flipping horizontally, vertically for more variation (only for training)
  ToTensor(): coverts 0-255 to 0-1
  Normalize: matching the pretraining preprocessing




In [7]:
from torchvision import transforms
train_transform = transforms.Compose([
transforms.Resize((224, 224)),
transforms.RandomHorizontalFlip(),
transforms.RandomVerticalFlip(),
transforms.ToTensor(),   #0-1
transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
transforms.Resize((224, 224)),
transforms.ToTensor(),
transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [8]:
from torchvision import datasets
from torch.utils.data import random_split

full_dataset = datasets.ImageFolder(root='/content/EuroSAT_RGB', transform=train_transform)

train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_set, val_set, test_set = random_split(full_dataset, [train_size, val_size, test_size])

Data Loading:
Loading the data in small batches for easier processing.

Freezing every layer except the last layer of the ResNet50 model so that we have the desired output=10

In [9]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_set, batch_size=32, shuffle=True,num_workers=2,
    pin_memory=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False,num_workers=2,
    pin_memory=True)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)

In [10]:
import torchvision.models as models
import torch.nn as nn
model = models.resnet50(pretrained=True)
for param in model.parameters():
  param.requires_grad = False
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)
model = model.to('cuda')

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 198MB/s]


training the model

In [ ]:
import torch
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to('cuda')
        labels = labels.to('cuda')
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to('cuda')
            labels = labels.to('cuda')
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    val_accuracy = correct / total
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}, Val Accuracy: {val_accuracy:.4f}')

Epoch 1/10, Loss: 0.4730, Val Accuracy: 0.9241
Epoch 2/10, Loss: 0.2818, Val Accuracy: 0.9290
Epoch 3/10, Loss: 0.2635, Val Accuracy: 0.9282
Epoch 4/10, Loss: 0.2404, Val Accuracy: 0.9342


Fine tuning the model


In [ ]:
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
import torch
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
num_epochs = 5
best_acc = 0
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to('cuda')
        labels = labels.to('cuda')
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to('cuda')
            labels = labels.to('cuda')
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    val_accuracy = correct / total
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}, Val Accuracy: {val_accuracy:.4f}')

    if val_accuracy > best_acc:
        best_acc = val_accuracy
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "accuracy": best_acc,
        }, "best_model.pth")
        print(f"✅ Saved Best Model (Epoch {epoch+1})")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to('cuda')

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

In [ ]:
class_names = [
    'AnnualCrop',
    'Forest',
    'HerbVeg',
    'Highway',
    'Industrial',
    'Pasture',
    'PermCrop',
    'Residential',
    'River',
    'SeaLake'
]

print(classification_report(
    all_labels,
    all_preds,
    target_names=class_names
))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10,8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
import cv2
!pip install ultralytics
from ultralytics import YOLO

# Load YOLOv8 model
model = YOLO("yolov8n.pt")


def detect_objects(frame):
    results = model(frame)
    detected_objects = []

    for r in results:
        for box in r.boxes:
            class_id = int(box.cls[0])  # Get class ID
            confidence = box.conf[0].item()  # Confidence score

            if confidence > 0.5:
                label = model.names[class_id]
                detected_objects.append(label)

                # Draw bounding box
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    return frame, detected_objects


def main():
    cap = cv2.VideoCapture(0)  # Open webcam

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame, detected_objects = detect_objects(frame)
        cv2.imshow("AI Vision", frame)
        if detected_objects:
            print("Detected objects:", detected_objects)
        key = cv2.waitKey(1) & 0xFF

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()